# ScGG on the LUNA mouse primary motor cortex benchmark (Figure 3)

Trains ScGG on Mouse 1 (33 slices, 158,379 cells) and evaluates on Mouse 2
(31 slices, 118,036 cells), reporting the exact LUNA metrics so we can
compare to the paper's 44.8% headline.

- **Data**: `/nfs/team361/sb75/DATASETS/silver/mmc_luna/`
  (64 per-slice h5ad files; auto-discovered by `load_luna_cortex`).
- **Metrics**: `scgg.evaluation.luna_metrics` — bit-equivalent to LUNA's
  `metrics/evaluation_statistics.py`.
- **Headline**: mean across 31 test slices of the per-slice median
  per-cell Spearman of pairwise-distance rows. LUNA paper: **0.448**.

See `../README.md` for full details on the setup and metric definitions.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os, logging
from pathlib import Path

# Adjust this to wherever you installed scgg (or `pip install -e .` first).
SCGG_SRC = Path('/path/to/scgg_claude_project/scgg/src')
if SCGG_SRC.exists() and str(SCGG_SRC) not in sys.path:
    sys.path.insert(0, str(SCGG_SRC))

# Make the benchmark script importable.
SCRIPTS = SCGG_SRC.parent / 'scripts'
if SCRIPTS.exists() and str(SCRIPTS.parent) not in sys.path:
    sys.path.insert(0, str(SCRIPTS.parent))

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(name)s: %(message)s')

import torch
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available(), '| GPUs:', torch.cuda.device_count())

## Sanity-check the data directory

Confirm the 33+31 LUNA split is on disk before training. The loader
discovers the split from filenames matching
`mmc_mouse{1,2}_slice{N}.h5ad`.

In [ ]:
DATA_DIR = '/nfs/team361/sb75/DATASETS/silver/mmc_luna'

from scgg.data.luna_cortex import _enumerate_slice_files
files = _enumerate_slice_files(Path(DATA_DIR))
n_train = sum(m == 1 for m, _, _ in files)
n_test  = sum(m == 2 for m, _, _ in files)
print(f'Train (Mouse 1) slices: {n_train}  (LUNA paper: 33)')
print(f'Test  (Mouse 2) slices: {n_test}   (LUNA paper: 31)')
assert n_train == 33 and n_test == 31, 'Split does not match LUNA Figure 3 expectations'

## Run the benchmark

This will:
1. Load Mouse 1 (train) and Mouse 2 (test) — applies log1p + scale by default.
2. Build ScGG in contrastive (PinSage-style) mode and train for `epochs`.
3. Generate a d=32 metric embedding for each test slice.
4. Compute Spearman / precision / RSSD per slice using LUNA's exact formulas.
5. Aggregate (mean of per-slice medians = LUNA headline) and write
   `per_slice_metrics.csv`, `aggregate_metrics.json`, frozen `config.yaml`,
   `benchmark.log`, and `checkpoints/` to `output_dir`.

In [ ]:
from scripts.run_luna_cortex_benchmark import run_benchmark

# Output landing zone. Default derived from DATA_DIR by the
# training script: {ARTIFACTS_ROOT}/<silver_dir_name>/model.
# Compute it explicitly here so later cells can read CSVs back.
OUTPUT_DIR = f'/nfs/team361/sb75/scgg-reproducibility/artifacts/{Path(DATA_DIR).name}/model'
agg = run_benchmark(
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
    # Override any defaults from scgg/configs/default.yaml here:
    epochs=200,
    batch_size=8192,
    lr=1e-3,
    seed=42,
    val_fraction=0.1,
    contact_percentile=0.01,
    compute_rssd=True,
    wandb=True,
    wandb_run_name='luna_cortex_v1',
)

headline = agg.get('spearman_mean_of_medians')
print()
print(f'ScGG headline (mean of per-slice median Spearman): {headline:.4f}')
print(f'LUNA paper headline:                              0.4480')
print(f'Delta: {(headline - 0.448) * 100:+.2f} percentage points')

## Per-slice breakdown

In [ ]:
import pandas as pd
df = pd.read_csv(Path(OUTPUT_DIR) / 'per_slice_metrics.csv')
df = df.sort_values('spearman_per_cell_median', ascending=False).reset_index(drop=True)
df[['section_label', 'n_cells',
    'spearman_per_cell_median', 'spearman_per_cell_mean',
    'precision', 'f1', 'absolute_rssd', 'mean_rssd']]

## Comparison vs LUNA paper baselines

Figure 3b in the LUNA paper reports (mean of per-slice median Spearman):

| Method       | Spearman |
|--------------|----------|
| LUNA         | 0.448    |
| CeLEry       | 0.224    |
| novoSpaRc    | 0.139    |
| CytoSPACE    | 0.117    |
| Tangram      | 0.099    |
| **ScGG (ours)** | (see above) |

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

labels = ['LUNA\n(paper)', 'CeLEry', 'novoSpaRc', 'CytoSPACE', 'Tangram', 'ScGG\n(ours)']
values = [0.448, 0.224, 0.139, 0.117, 0.099, headline]
colors = ['tab:orange'] * 5 + ['tab:blue']

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(np.arange(len(labels)), values, color=colors)
ax.set_xticks(np.arange(len(labels)))
ax.set_xticklabels(labels)
ax.set_ylabel("Spearman's rank correlation\n(mean of per-slice medians)")
ax.set_title('LUNA Figure 3 reproduction: cross-animal cortex test')
ax.axhline(0.448, ls='--', color='gray', lw=0.8)
fig.tight_layout()
fig.savefig(Path(OUTPUT_DIR) / 'figure3_comparison.png', dpi=150)
plt.show()